# Model evaluation

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

Model evaluation answers one question : **how close is this model to the real animals?** It runs
one or more candidate models under conditions matched to a reference dataset, computes the same
kinematic metrics on both, and reports the distance between them.

It is worth being precise about what that means, because the answer is never a single number. A
model can reproduce crawling speed perfectly and turn far too often; another can get the turn rate
right and crawl too slowly. Evaluation therefore compares *distributions* of many metrics, grouped
into categories, and the result is a table of errors rather than a verdict.

This is also the measurement that the genetic algorithm optimizes - which is why this notebook
comes before it.

**What you will be able to do afterwards**

- Point an evaluation at a reference dataset and a list of candidate models.
- Say what the four metric categories measure and choose your own metrics.
- Understand the normalization and evaluation modes, and why more than one is reported.
- Produce the comparison figures for models and for the underlying datasets.

**Prerequisites** : [Your first simulation](../1_getting_started/single_simulation.ipynb).

**Cost** : the demo run is 5 larvae for 0.5 simulated minutes per model, but it is off by default
because it also enriches the resulting datasets.

**Switches in this notebook**

| switch | default | what it turns on |
|---|---|---|
| `RUN_EVAL_DEMO` | `False` | the evaluation simulation itself |
| `RUN_PLOTS_DEMO` | `False` | the result and model comparison figures |

## Setup

In [1]:
%matplotlib inline

%load_ext param.ipython

import larvaworld as lw
from larvaworld.lib import reg
from larvaworld.lib.sim.model_evaluation import EvalConf, EvalRun

lw.VERBOSE = 1

# Tutorial safety switches
RUN_EVAL_DEMO = False  # runs the simulations and enriches the datasets
RUN_PLOTS_DEMO = False  # produces the comparison figures

DEMO_REF_ID = "exploration.30controls"
DEMO_MODEL_IDS = ["explorer", "navigator"]
DEMO_N = 5  # larvae per model
DEMO_DURATION_MIN = 0.5  # >= 0.33 min so that the 20 s metrics are defined
DEMO_SCREEN_KWS = {}  # headless

Welcome to the param IPython extension! (https://param.holoviz.org/)
Available magics: %params


Initializing larvaworld registry


Registry configured!


## Section 1 : What an evaluation needs

Three things :

1. **A reference dataset** - real recordings, identified by a reference ID or by the directory it
   lives in. Everything else is matched to it : the arena, the timestep, the duration, and where
   available the body length distribution of the animals.
2. **Candidate models** - one or more model IDs. Each becomes one larva group in the simulation, so
   the models are compared under identical conditions rather than in separate runs.
3. **Evaluation metrics** - which measured quantities count, and optionally which parts of the
   stride cycle curve should be fitted.

`EvalConf` holds all three.

In [2]:
%params EvalConf

### The reference datasets available

A reference dataset is an imported experimental dataset that has been registered under an ID. The
package ships with a few; anything you import yourself with a `refID` joins this list.

In [3]:
refIDs = reg.conf.Ref.confIDs
print(f"{len(refIDs)} reference datasets :")
print(refIDs)
print()
print("Default used across the tutorials :", reg.default_refID)

11 reference datasets :
['Chris.larvae_single', 'DeepLabCut.TopDown-2023-07-05', 'DeepLabCut.TopDown-2024-02-17', 'FeedingState.Fed', 'FeedingState.Starved', 'FeedingState.Sucrose', 'FreeExploration.pooled', 'FreeExploration.single_dish', 'exploration.30controls', 'exploration.dish01', 'exploration.dish02']

Loaded existing conf larvae_single
Loaded stored reference dataset : Chris.larvae_single
Loaded existing conf TopDown-2023-07-05
Loaded stored reference dataset : DeepLabCut.TopDown-2023-07-05
Loaded existing conf TopDown-2024-02-17
Loaded stored reference dataset : DeepLabCut.TopDown-2024-02-17
Loaded existing conf Fed
Loaded stored reference dataset : FeedingState.Fed
Loaded existing conf Starved
Loaded stored reference dataset : FeedingState.Starved
Loaded existing conf Sucrose
Loaded stored reference dataset : FeedingState.Sucrose
Loaded existing conf pooled
Loaded stored reference dataset : FreeExploration.pooled


Loaded existing conf single_dish
Loaded stored reference dataset : FreeExploration.single_dish
Loaded existing conf 30controls
Loaded stored reference dataset : exploration.30controls
Loaded existing conf dish01
Loaded stored reference dataset : exploration.dish01
Loaded existing conf dish02
Loaded stored reference dataset : exploration.dish02
Default used across the tutorials : exploration.30controls


### The candidate models

Any stored model can be evaluated. `explorer` and `navigator` are a useful pair for a
free-exploration reference : they share the locomotory machinery, and the second adds olfactory
navigation on top. Against a dataset recorded without any odor they should therefore be close to
each other - which is itself a sanity check on the evaluation.

In [4]:
mIDs = reg.conf.Model.confIDs
print(f"{len(mIDs)} stored models. The two used below :")
for mID in DEMO_MODEL_IDS:
    print(f"  {mID!r} : {'stored' if mID in mIDs else 'MISSING'}")

612 stored models. The two used below :
  'explorer' : stored
  'navigator' : stored


### The metrics

The default metric set is grouped into four categories, and the grouping is the point : a model is
reported as good or bad *per category*, so you can see where it fails.

| category | what it captures |
|---|---|
| **angular kinematics** | bending, angular velocity and acceleration, turn amplitude |
| **spatial displacement** | path length, crawling speed, stride length, dispersal |
| **temporal dynamics** | stride frequency, run and pause durations and their ratios |
| **stride cycle** | the shape of the within-stride velocity and bending curves |

You can replace the whole dictionary with your own selection. The keys are short parameter codes;
`reg.getPar` translates between them and readable labels.

In [5]:
ev_conf = EvalConf(refID=DEMO_REF_ID)

for category, ks in ev_conf.eval_metrics.items():
    print(f"{category} ({len(ks)}) :")
    print(f"   {ks}")

Loaded existing conf 30controls


Loaded stored reference dataset : exploration.30controls
angular kinematics (8) :
   ['run_fov_mu', 'pau_fov_mu', 'b', 'fov', 'foa', 'rov', 'roa', 'tur_fou']
spatial displacement (9) :
   ['cum_d', 'run_d', 'str_c_l', 'v_mu', 'pau_v_mu', 'run_v_mu', 'v', 'a', 'dsp_0_40_max']
temporal dynamics (6) :
   ['fsv', 'ffov', 'run_t', 'pau_t', 'run_tr', 'pau_tr']
stride cycle (9) :
   ['str_d_mu', 'str_d_std', 'str_sv_mu', 'str_fov_mu', 'str_fov_std', 'str_N', 'str_t', 'str_d', 'str_sd']
tortuosity (4) :
   ['tor5', 'tor20', 'tor5_mu', 'tor20_mu']


### Normalization and evaluation modes

Two further choices decide how the errors are reported :

- **`norm_modes`** - `raw` keeps each metric in its own units, so the categories are not
  comparable; `minmax` rescales them so they are; `std` standardizes them.
- **`eval_modes`** - `pooled` compares the pooled distributions of model and reference;
  `1:1` compares each simulated individual with one real individual; `1:pooled` compares each
  simulated individual against the pooled reference.

The defaults report both a raw and a minmax view of the pooled comparison, which is usually what
you want to look at first.

In [6]:
print("norm_modes :", ev_conf.norm_modes)
print("eval_modes :", ev_conf.eval_modes)

norm_modes : ['raw', 'minmax']
eval_modes : ['pooled']


## Section 2 : Running it

`EvalRun` is the launcher. It reads the reference dataset first, so the timestep and duration of
the simulations are already matched when the run starts - you do not have to set them yourself.

In [7]:
%params EvalRun

In [8]:
kws = {
    "refID": DEMO_REF_ID,
    "modelIDs": DEMO_MODEL_IDS,
    "experiment": "dish",
    "N": DEMO_N,
    "duration": DEMO_DURATION_MIN,
    "screen_kws": DEMO_SCREEN_KWS,
}

r = EvalRun(**kws)

print(f"Reference : {r.target.id!r} with {r.target.config.N} larvae")
print(f"Models    : {r.modelIDs}")
print(f"dt        : {r.dt} s")

Loaded existing conf 30controls


Loaded stored reference dataset : exploration.30controls
Reference : 'experiment' with 30 larvae
Models    : ['explorer', 'navigator']
dt        : 0.0625 s


In [9]:
if RUN_EVAL_DEMO:
    r.simulate()
    print("Error tables produced for :", list(r.error_dicts.keys()))
else:
    print("Set RUN_EVAL_DEMO = True to run the evaluation.")

Set RUN_EVAL_DEMO = True to run the evaluation.


## Section 3 : Reading the result

Two families of figures come out of an evaluation :

- **`plot_results`** compares the *datasets* - the simulated ones against the reference - on the
  evaluated metrics, plus the error tables themselves.
- **`plot_models`** compares the *models* to each other, which is what tells you which candidate to
  keep.

In [10]:
if RUN_PLOTS_DEMO and RUN_EVAL_DEMO:
    r.plot_results(show=False)
    r.plot_models(show=False)
    print(f"Figures written under {r.plot_dir}")
else:
    print("Set RUN_EVAL_DEMO and RUN_PLOTS_DEMO to True to produce the figures.")

Set RUN_EVAL_DEMO and RUN_PLOTS_DEMO to True to produce the figures.


## Where to go next

- [Genetic algorithm optimization](genetic_algorithm_optimization.ipynb) - the same error, used as
  a fitness function to search a model's parameter space.
- [Worked example: turner noise](ga_turner_noise_optimization.ipynb) - evaluation before and after
  an optimization, on a single module.
- [Importing experimental data](../2_experimental_data/import_datasets.ipynb) - how to turn your
  own recordings into a reference dataset.
- Reference : [Model evaluation](../../working_with_larvaworld/model_evaluation.md).